# Polars y Plottly

## 1. Objetivo

El objetivo de la presente sección será trabajar con

1) `Polars`, la cuales es una alternativa de trabajo a Pandas, enfocada a computo intensivo y datos de mayor volumen. Véase https://pola.rs y https://docs.pola.rs/api/python/stable/reference/index.html

2) El módulo `Plotly Express` de la librería `Plotly` de Python (https://plotly.com/python/). Ésta es una librería para realizar gráficos interactivos en Python de amplio espectro.

## 2. Librerias de trabajo

In [1]:
pip install polars plotly numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import plotly.express as px
import numpy as np
import polars as pl

import warnings
warnings.filterwarnings('ignore')

In [73]:
import polars as pl
import datetime as dt

df = pl.DataFrame(
    {
        "name": ["Alice Archer", "Ben Brown", "Chloe Cooper", "Daniel Donovan"],
        "birthdate": [
            dt.date(1997, 1, 10),
            dt.date(1985, 2, 15),
            dt.date(1983, 3, 22),
            dt.date(1981, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1],
        "height": [1.56, 1.77, 1.65, 1.75],
    }
)

print(df)

shape: (4, 4)
┌────────────────┬────────────┬────────┬────────┐
│ name           ┆ birthdate  ┆ weight ┆ height │
│ ---            ┆ ---        ┆ ---    ┆ ---    │
│ str            ┆ date       ┆ f64    ┆ f64    │
╞════════════════╪════════════╪════════╪════════╡
│ Alice Archer   ┆ 1997-01-10 ┆ 57.9   ┆ 1.56   │
│ Ben Brown      ┆ 1985-02-15 ┆ 72.5   ┆ 1.77   │
│ Chloe Cooper   ┆ 1983-03-22 ┆ 53.6   ┆ 1.65   │
│ Daniel Donovan ┆ 1981-04-30 ┆ 83.1   ┆ 1.75   │
└────────────────┴────────────┴────────┴────────┘


## 3. Lectura de datos

Primero nos encargaremos de leer los datos, indicando a Python donde se encuentra la carpeta que se aloja los datos y los nombres de los archivos relevantes para el análisis.

In [9]:
# Primero indicamos la ruta a la carpeta de de tu computadora
# donde se ubican los datos con los que vamos a trabajar
# Ejemplo: "C:\Usuarios\[tu nombre]\Descargas"

DATA_PATH = "/Users/cesar/sandbox/ai_programming_foundations/data"


Además de la data procesada, leeremos el archivo **brasil_geodata.json**, el cual es información geográfica de los estados de Brasil que será útil para nuestro análisis. Dicho archivo es una versión procesada del archivo `Brasil.json` de Kaggle (https://www.kaggle.com/code/kerneler/starter-brazil-states-geojson-ca176cdb-a).

Adicionalmente, para enriquecer el análisis añadirá al archivo `brasil_regions.csv` que contiene una clasificiación de los estados de Brasil en 4 regiones geográficas (`north`, `northeast`, `south` y `center-west`):

In [10]:
FILE_GEODATA = 'brasil_geodata.json'
FILE_CONSOLIDATED_DATA = 'oilst_processed.csv'
FILE_REGIONS = 'brasil_regions.csv'

In [11]:
# Cargar archivo datos geográficos de Brasil
with open(os.path.join(DATA_PATH, FILE_GEODATA), 'r') as f:
    geojson = json.load(f)


In [12]:
regions = pl.read_csv(
    os.path.join(DATA_PATH, FILE_REGIONS),
)

In [18]:
# cargaamos datos de órdenes procesadas
columns_dates = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

oilst = pl.read_csv(
    os.path.join(DATA_PATH, FILE_CONSOLIDATED_DATA),
   try_parse_dates=True
)


In [20]:
oilst.sample(10)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,distance_distribution_center,year,month,quarter,year_month,delta_days,delay_status,total_products,total_sales,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,abbreviation,state_name
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],date,f64,i64,i64,str,str,f64,str,f64,f64,str,i64,str,str,i64,f64,f64,str,str,str,str
"""4f3e39331cc2ae23851f526e268a88…","""255076106ffe1cb62993b4f96aa67e…","""delivered""",2018-02-26 15:15:43,2018-02-26 15:51:41,2018-02-28 22:12:43,2018-04-10 21:27:24,2018-03-26,804.96,2018,2,"""2018Q1""","""2018-02""",15.894028,"""long_delay""",1.0,104.9,"""1d58387408a543a2eb1d8561965cb0…",48770,"""teofilandia""","""BA""",48770,-11.473213,-39.00347,"""teofilandia""","""BA""","""BA""","""Bahia"""
"""a7dfdfebf93719bd1ad61e569a852c…","""62286ca771b7e7fc40c28aca1a8a42…","""delivered""",2018-04-27 12:04:24,2018-04-28 03:15:32,2018-05-09 14:14:00,2018-05-15 13:54:33,2018-05-29,86.07,2018,4,"""2018Q2""","""2018-04""",-13.420451,"""on_time""",1.0,179.9,"""54b860479701b0143d801783b6e87e…",23934,"""angra dos reis""","""RJ""",23934,-22.969506,-44.303397,"""angra dos reis""","""RJ""","""RJ""","""Rio de Janeiro"""
"""e1bf461767c45a8f0abf0a641e6bd4…","""965d83ab80a4b9913126a2856bb81e…","""delivered""",2017-05-15 16:02:25,2017-05-15 16:10:17,2017-05-17 11:20:03,2017-05-24 14:18:50,2017-06-08,28.4,2017,5,"""2017Q2""","""2017-05""",-14.403588,"""on_time""",1.0,82.5,"""e476843c331c4261a33844b54d0d34…",71680,"""brasilia""","""DF""",71680,-15.843093,-47.822252,"""brasília""","""DF""","""DF""","""Distrito Federal"""
"""1f04f45771eed910a8d58e5c526046…","""eaf673c8640de260cb58679e0e159c…","""delivered""",2018-02-19 13:42:51,2018-02-19 13:50:57,2018-02-23 01:03:49,2018-03-07 22:52:10,2018-03-13,1.66,2018,2,"""2018Q1""","""2018-02""",-5.047106,"""on_time""",2.0,46.96,"""e03684ee60e4f8113ddbe25a1a33d4…",30260,"""belo horizonte""","""MG""",30260,-19.924397,-43.911676,"""belo horizonte""","""MG""","""MG""","""Minas Gerais"""
"""86c4303307d8b4c57e6359d46e9384…","""12c909b646762b18b93b1d61d4efcf…","""delivered""",2017-06-25 16:16:00,2017-06-25 16:25:12,2017-06-26 13:07:27,2017-07-06 15:06:45,2017-07-19,16.55,2017,6,"""2017Q2""","""2017-06""",-12.370312,"""on_time""",1.0,49.9,"""51c2f039f08424330a6bbba4c9949d…",13348,"""indaiatuba""","""SP""",13348,-23.128515,-47.240606,"""indaiatuba""","""SP""","""SP""","""São Paulo"""
"""bc894fc83cb90dae37405580d66412…","""81726f29e5d81a08428ffbce397286…","""delivered""",2018-02-01 21:02:51,2018-02-06 06:29:50,2018-02-07 10:16:56,2018-02-14 13:55:50,2018-02-26,38.57,2018,2,"""2018Q1""","""2018-02""",-11.41956,"""on_time""",1.0,503.34,"""cb2c9cb3fc3f2f035deaada37f3231…",21020,"""rio de janeiro""","""RJ""",21020,-22.830107,-43.277549,"""rio de janeiro""","""RJ""","""RJ""","""Rio de Janeiro"""
"""388d1145ac72ea94751f572092f528…","""c7f397d8574d2a4a48fcaa7d6cd7a0…","""delivered""",2018-02-13 10:06:18,2018-02-13 10:40:26,2018-02-16 01:55:02,2018-02-21 21:49:12,2018-02-26,23.3,2018,2,"""2018Q1""","""2018-02""",-4.090833,"""on_time""",1.0,29.9,"""e99bcd86a7d27f3c7eb57ca29ed9f8…",8683,"""suzano""","""SP""",8683,-23.529357,-46.314429,"""suzano""","""SP""","""SP""","""São Paulo"""
"""ff3d74e7a6f632103cf96ffeb8ee9a…","""f1cf04d23b4eb09ac829c8504b6a4e…","""delivered""",2018-07-12 22:14:53,2018-07-12 22:30:11,2018-07-24 16:07:00,2018-07-25 20:44:32,2018-07-30,2.26,2018,7,"""2018Q3""","""2018-07""",-4.135741,"""on_time""",1.0,154.91,"""0f4234c79ad61fe9ab42470466c3ce…",7183,"""guarulhos""","""SP""",7183,-23.44549,-46.458276,"""guarulhos""","""SP""","""SP""","""São Paulo"""
"""559eea5a72341a4c82dbce9884277c…","""5fd5cb96f515996e88d84c61012d97…","""delivered""",2017-11-17 10:24:03,2017-11-17 10:51:09,2017-11-20 17:38:35,2017-11-30 

In [22]:
# Agregamos la columna de región para los estadios de Brasil
oilst = oilst.join(regions[['abbreviation', 'region']], on='abbreviation', how='left')

In [24]:
oilst.describe()


statistic,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,distance_distribution_center,year,month,quarter,year_month,delta_days,delay_status,total_products,total_sales,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,abbreviation,state_name,region
str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,f64,str,f64,f64,str,f64,str,str,f64,f64,f64,str,str,str,str,str
"""count""","""99441""","""99441""","""99441""","""99441""","""99281""","""97658""","""96476""","""99441""",96470.0,99441.0,99441.0,"""99441""","""99441""",96476.0,"""99441""",98666.0,98666.0,"""99441""",99441.0,"""99441""","""99441""",99163.0,99163.0,99163.0,"""99163""","""99163""","""99163""","""99163""","""99163"""
"""null_count""","""0""","""0""","""0""","""0""","""160""","""1783""","""2965""","""0""",2971.0,0.0,0.0,"""0""","""0""",2965.0,"""0""",775.0,775.0,"""0""",0.0,"""0""","""0""",278.0,278.0,278.0,"""278""","""278""","""278""","""278""","""278"""
"""mean""",null,null,null,"""2017-12-31 08:43:12.776581""","""2017-12-31 18:35:24.098800""","""2018-01-04 21:49:48.138278""","""2018-01-14 12:09:19.035542""","""2018-01-24 03:08:37.730111""",389.921076,2017.539838,6.03222,null,null,-11.17912,null,1.141731,137.754076,null,35137.474583,null,null,35057.887176,-21.19293,-46.17619,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,13123.006121,0.505007,3.232999,null,null,10.186113,null,0.538452,210.645145,null,29797.938996,null,null,29784.314664,5.620584,4.066202,null,null,null,null,null
"""min""","""00010242fe8c5a6d1ba2dd792cb162…","""00012a2ce6f8dcda20d059ce984917…","""approved""","""2016-09-04 21:15:19""","""2016-09-15 12:16:38""","""2016-10-08 10:34:01""","""2016-10-11 13:46:32""","""2016-09-30""",0.01,2016.0,1.0,"""2016Q3""","""2016-09""",-146.016123,"""long_delay""",1.0,0.85,"""0000366f3b9a7992bf8c76cfdf3221…",1003.0,"""abadia dos dourados""","""AC""",1003.0,-36.605374,-72.666706,"""abadia dos dourados""","""AC""","""AC""","""Acre""","""center-west"""
"""25%""",null,null,null,"""2017-09-12 14:46:19""","""2017-09-12 23:24:16""","""2017-09-15 22:27:50""","""2017-09-25 22:07:34""","""2017-10-03""",23.51,2017.0,3.0,null,null,-16.244375,null,1.0,45.9,null,11347.0,null,null,11320.0,-23.588296,-48.0953,null,null,null,null,null
"""50%""",null,null,null,"""2018-01-18 23:04:36""","""2018-01-19 11:36:13""","""2018-01-24 16:12:38""","""2018-02-02 19:28:30""","""2018-02-15""",50.39,2018.0,6.0,null,null,-11.948924,null,1.0,86.9,null,24416.0,null,null,24350.0,-22.926905,-46.630764,null,null,null,null,null
"""75%""",null,null,null,"""2018-05-04 15:42:16""","""2018-05-04 20:35:10""","""2018-05-08 13:38:00""","""2018-05-15 22:48:50""","""2018-05-25""",77.13,2018.0,8.0,null,null,-6.390093,null,1.0,149.9,null,58900.0,null,null,58407.0,-20.146615,-43.602775,null,null,null,null,null
"""max""","""fffe41c64501cc87c801fd61db3f62…","""ffffe8b65bbe3087b653a978c870db…","""unavailable""","""2018-10-17 17:30:18""","""2018-09-03 17:40:06""","""2018-09-11 19:48:28""","""2018-10-17 13:22:46""","""2018-11-12""",1.3497e6,2018.0,12.0,"""2018Q4""","""2018-10""",188.975081,"""short_delay""",21.0,13440.0,"""ffffd2657e2aad2907e67c3e9daecb…",99990.0,"""zortea""","""TO""",99990.0,42.184003,-8.577855,"""óleo""","""TO""","""TO""","""Tocantins""","""south"""


A su vez transformaremos el nombre de la ciudades a un formato de título, es decir, las primeras letras de las palabras estarán en mayúsculas.

In [26]:
oilst['geolocation_city']

geolocation_city
str
"""sao paulo"""
"""barreiras"""
"""vianopolis"""
"""sao goncalo do amarante"""
"""santo andre"""
…
"""são josé dos campos"""
"""praia grande"""
"""nova vicosa"""


In [28]:
oilst = oilst.with_columns(
    pl.col("geolocation_city").str.to_titlecase().alias("geolocation_city")
)

In [29]:
oilst['geolocation_city']

geolocation_city
str
"""Sao Paulo"""
"""Barreiras"""
"""Vianopolis"""
"""Sao Goncalo Do Amarante"""
"""Santo Andre"""
…
"""São José Dos Campos"""
"""Praia Grande"""
"""Nova Vicosa"""


También definiremos un dataframe que contiene únicamente a las órdenes que tienen estatus de entrega completada, es decir, que satisfacen con la condición de que la columna `order_status` es igual al valor `delivered`.

In [31]:
delivered = oilst.filter(
    pl.col("order_status") == "delivered"
)

### 4. Usando el API de Plotly

Plotly es una librería de visualización de datos (https://plotly.com/python/) que te permite crear gráficos interactivos y con una estética llamativa a través de una interfaz sencilla para el usuario, que se basa en la libreria `D3.js` de Javascript pero con una interfaces en Python, R y otro.

Básicamente, con Plotly puedes tomar datos y crear gráficos de barras, líneas, áreas, dispersión, histogramas, y también algunos en 3D. Lo que hace que Plotly sea especial es que puedes personalizar tus gráficos fácilmente y hacerlos interactivos, lo que significa que puedes hacer clic en los elementos del gráfico para obtener más información o incluso modificarlos en tiempo real.

Por ejemplo, si tienes un conjunto de datos de ventas de diferentes productos, puedes usar Plotly para crear un gráfico de barras que muestre las ventas de cada producto en un período determinado. Luego, si un usuario hace clic en una barra específica, Plotly puede mostrar información detallada sobre esa venta en particular.

Esta sección exploraremos particularmente el módulo de `Plotly Express`, aprovechando que la sintaxis es muy similar a la que se emplea en Seaborn, pues es compatible con dataframes de Pandas.


### 4.1 Análisis en el tiempo de la cantidad de órdenes de acuerdo a si se entregaron o no a tiempo.

Para comenzar, podemos explorar nuevamente la cantidad de ordenes en función de si llegaron en tiempo al domicilio del cliente.

En este caso, primero calcularemos los valores agregados de las ordenes por dicho estatus.

In [34]:
# Calcula la cantidad de ordenes en el tiempo
orders_time = delivered.group_by('year_month').agg(
    pl.col('order_id').count().alias('orders')
).sort('year_month') # Add .sort() for proper time series ordering

# Crea una variable temporal en texto para graficar
orders_time = orders_time.with_columns(
    pl.col('year_month').cast(pl.Utf8).alias('period')
)

Visualmente ello construye la siguiente tabla:

In [35]:
orders_time.tail()

year_month,orders,period
str,u32,str
"""2018-04""",6798,"""2018-04"""
"""2018-05""",6749,"""2018-05"""
"""2018-06""",6099,"""2018-06"""
"""2018-07""",6159,"""2018-07"""
"""2018-08""",6351,"""2018-08"""


Para realizar un gráfico de barras interactivo, basta usar la función `.bar` (https://plotly.com/python/bar-charts/), cuya sintaxis es análoga a la de Seaborn

In [36]:
# Crea la visualizacion
fig = px.bar(
    orders_time,
    x="period",
    y="orders",
    title='Fig.1 Número de órdenes de Oilst'
)

# Muestra la figura
fig.show()

Notas:

    * Si se pasa el mouse sobre la visualización, se verá que esta despliega la información interactiva de los valores de la grafica,
    * Ademas permite hacer zoom, crear recortes, seleccionar regiones con un lazo, moverse y otras.

Similarmente a lo que se exploró en Seaborn, las visualizaciones se pueden segmentar usando otras variables. En este caso introduciremos un segmentación de acuerdo a si la orden se entregó o no en tiempo.

In [38]:
# Calcula la cantidad de ordenes en el tiempo agrupadas por estado de retraso
orders_time_delay_status = (
    delivered.group_by(['year_month', 'delay_status'])
    .agg(
        # 1. Count the order_id column
        # 2. Alias the result to 'orders'
        pl.col('order_id').count().alias('orders')
    )
    .sort(['year_month', 'delay_status']) # Good practice to sort results
)

# Crea una variable temporal en texto para graficar
orders_time_delay_status = orders_time_delay_status.with_columns(
    # Cast the year_month column (likely a date/int) to a string (Utf8)
    pl.col('year_month').cast(pl.Utf8).alias('period')
)

Esto nos arroja la tabla:

In [39]:
orders_time_delay_status

year_month,delay_status,orders,period
str,str,u32,str
"""2016-09""","""long_delay""",1,"""2016-09"""
"""2016-10""","""long_delay""",1,"""2016-10"""
"""2016-10""","""on_time""",262,"""2016-10"""
"""2016-10""","""short_delay""",2,"""2016-10"""
"""2016-12""","""on_time""",1,"""2016-12"""
…,…,…,…
"""2018-07""","""on_time""",5880,"""2018-07"""
"""2018-07""","""short_delay""",121,"""2018-07"""
"""2018-08""","""long_delay""",232,"""2018-08"""


Dicha información se puede analizar mejor a través del siguiente gráfico de barras:

In [40]:
# Crea la visualizacion
fig = px.bar(
    orders_time_delay_status,
    x="period",
    y="orders",
    color='delay_status',
    title='Fig.2 Número de órdenes de Oilst por tipo de entrega'
)

# Muestra la figura
fig.show()

Notas:

    * El gráfico anterior se puede filtra activando los colores del cuadro superior derecho para esconder o mostrar cada grupo. Prueba dando click en el cuadrado rojo con la etiqueta `on_time`
    * También se pueden desagrupar las barras usando el parámetro ` barmode='group'`

**Preguntas:**

* ¿Cuándo sucede el periodo mas alto de retrazos prolongados?
* ¿Existe alguna relación con el incremento de órdenes totales que el e-commerce empezó a recibir o no?

### 4.2 Análisis en el tiempo de las ventas de acuerdo a si se entregaron o no a tiempo.

Otra vertiente de análisis, es por supuesta la cantidad de ventas en función de si las órdenes llegaron en tiempo al domicilio del cliente.

Nuevamente calcularemos los valores agregados de las órdenes por dicho estatus.

In [42]:
# Calculate the sum of 'total_sales' grouped by 'quarter' and 'delay_status'
sales_time = (
    delivered.group_by(['quarter', 'delay_status'])
    .agg(
        # Sum the 'total_sales' column and alias the result to 'total_sales'
        pl.col('total_sales').sum().alias('total_sales')
    )
    .sort(['quarter', 'delay_status']) # Sort for a predictable order
)

# Convert the 'quarter' column to a string type for plotting
sales_time = sales_time.with_columns(
    pl.col('quarter').cast(pl.Utf8).alias('quarter')
)

En este caso, aprovecharemos para introdución las gráficas de áreas de Plotly, que esencialmente se trata de series de tiempo con sobreados que permite entender la magnitun de una cantidad en el tiempo como el área bajo un curva.

In [43]:
fig = px.area(
    sales_time,
    x="quarter",
    y="total_sales",
    color="delay_status",
    title='Fig.3 Total de ventas de órdenes de Oilst por tipo de entrega'
    )
fig.show()

Del gráfico es claro que desde el ultimo trimestre de 2017 y hasta el segundo trimestre de 2018, la compañia enfrentó un crecimiento formidable en ventas, lo que podría indicar un sobre esfuerzo de los procedimientos logísticos al tener que lidiar con más pedidos de lo normal.

**Pregunta:**

* ¿Existe alguna relación entre este hecho y los retrasos reportados en las entregas?

Ahora bien, aunque la gráfica anterior es bastante ilustrativa, para transmitir la magnitud ecónomica que representaría que todas las órdenes con retrazo cancelaran, que es el peor de los escenarios posibles, se necesita un gráfico similar pero que muestre los valores en ventas como proporciones, pues tales cantidades son más sencillas de entender.

Ahora se consolidará un gráfica de ese estilo:

    * Primero se cacularan los valores de ventas de acuerdo al estatus de llegada del pedido,
    * Luego se calcularan como proporciones dentro de cada trimestre, usando la normalización de la función `crosstab` de Pandas,
    * Después se podrán los datos en un formato alargado (https://pandas.pydata.org/docs/reference/api/pandas.melt.html), basicamente para poder graficar de forma más sencilla
    * Y finalmente se usará el api de área de Plotly

In [45]:
# 1. Group, Sum Sales, and Calculate Proportions (using Window Functions)
# This replaces the first two blocks of Pandas code (groupby and crosstab/normalize)
sales_time_delay_status_proportions = (
    delivered
    .group_by(['quarter', 'delay_status'])
    .agg(
        # Calculate the sum of sales for each (quarter, delay_status) group
        pl.col('total_sales').sum().alias('sales_by_status')
    )
    .with_columns(
        # Calculate the total sales for the entire quarter (Window Function)
        pl.col('sales_by_status').sum().over('quarter').alias('sales_by_quarter')
    )
    .with_columns(
        # Calculate the proportion (sales_by_status / sales_by_quarter) * 100
        (pl.col('sales_by_status') / pl.col('sales_by_quarter') * 100.0)
        .round(2)
        .alias('value_percent')
    )
    # Select and rename columns to match the output of your melt operation
    .select(
        pl.col('quarter').cast(pl.Utf8).alias('quarter'), # Cast to string for plotting
        pl.col('delay_status').alias('variable'),         # Renamed to 'variable'
        pl.col('value_percent').alias('value')            # Renamed to 'value'
    )
)

# The result is already in the 'long' format, similar to what pd.melt produces,
# but it's more direct and efficient.
sales_time_delay_status_tab_formated = sales_time_delay_status_proportions

In [51]:
sales_time_delay_status_tab_formated

quarter,variable,value
str,str,f64
"""2018Q1""","""on_time""",84.17
"""2016Q4""","""short_delay""",0.69
"""2017Q4""","""long_delay""",7.91
"""2018Q3""","""short_delay""",4.21
"""2017Q4""","""on_time""",88.68
…,…,…
"""2018Q2""","""on_time""",94.66
"""2017Q1""","""on_time""",95.85
"""2018Q1""","""short_delay""",3.95


In [52]:

fig = px.area(
    sales_time_delay_status_tab_formated,
    x="quarter",
    y="value",
    color="variable",
    title="Fig. 4 Proporción de ventas de Oilst que representan las órdenes por tipo de entrega"
    )
fig.show()

**Preguntas**

* Fuera de los primeros meses del e-commerce, ¿Cuándo exisitió una mayor proporción de ventas con retraso?
* ¿Cómo puede explicarse en relación con las etapas de crecimiento en ventas totales de la empresa?

### 5.1 Análisis regional sobre los retrasos en órdenes

Hasta ahora no se ha explorar la relación que existe entre la ubicación geográfica de los clientes con los retrasos en entrega. 

Abordaremos la relación entre ambos puntos usando visualizaciones interactivas. Para ellos se debe mencionar que se ha incorporado a los datos un clasificación de los estados de Brasil en regiones, descritar por la tabla siguiente:

In [54]:
print(oilst.columns)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'distance_distribution_center', 'year', 'month', 'quarter', 'year_month', 'delta_days', 'delay_status', 'total_products', 'total_sales', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'abbreviation', 'state_name', 'region']


In [58]:
oilst.select('region', 'abbreviation', 'state_name')\
    .unique().sort(['region', 'abbreviation']).drop_nulls()

region,abbreviation,state_name
str,str,str
"""center-west""","""DF""","""Distrito Federal"""
"""center-west""","""GO""","""Goiás"""
"""center-west""","""MT""","""MatoGrosso"""
"""north""","""AC""","""Acre"""
"""north""","""AM""","""Amazonas"""
…,…,…
"""northwest""","""SP""","""São Paulo"""
"""south""","""MS""","""MatoGrosso do Sul"""
"""south""","""PR""","""Paraná"""


Para aquellas ordenes con retrazos prolongados, las distribuciones tiempos de retrazo por región pueden visualar con diagramas de caja:

In [62]:
fig = px.box(delivered.filter(pl.col("delay_status") == 'long_delay'),
    x="region",
    y="delta_days",
    title="Fig. 5 Distribución de los tiempos de entrega de órdenes con retrazo, por región"
)

fig.show()


In [63]:
fig = px.violin(delivered.filter(
    pl.col("delay_status") == 'long_delay'
),
    x="region",
    y="delta_days",
    color="region",
    box=True, points="all",
    title="Fig. 6 Gráfico de violín de los tiempos de entrega de órdenes con retrazo, por región"
)

fig.show()

De lo anterior, se desprende que la región noreste tiene muchos valores atípicos en los tiempos de entrega. Esta es una hipótesis interesante de análisis.

Para complementarla, se puede segmentar aun más la visualización a nivel estado:

In [64]:
fig = px.box(delivered.filter(
    pl.col("delay_status") == 'long_delay'
),
    x="state_name",
    y="delta_days",
    color="region",
    title="Fig. 7 Distribución de los tiempos de entrega de órdenes con retrazo, por estado y región"
)

fig.show()

En los diagramas de caja anterior, se aprecia que los estados de Sao Paolo y Rio de Janeiro son los que tienen valores más llamativas de tiempos de entrega altos en esa región. Sin embargo, al desagregar los resultados también notramos que hay anomalís en estados como el Amazonas y Roraima.

### 5. 2 Visualizaciones Geográficas

Uno de lo puntos más interesantes de Ploty es la posibilidad de realizar gráficos completos usando data de otros sistemas, como los de origen geográfico.

En esta sección se mostrarán visualizaciones de los tiempos promedios por entrega en cada estado. Para ello se estimará el valor medio de los retrazos.

In [67]:
# Calcula el valor promedio de retrazos en el estado

delay_by_state = (
    delivered.filter(
        pl.col("delay_status") == 'long_delay'
    )
    .group_by(['state_name', 'geolocation_state'])
    .agg(
        # Calculate the mean of 'delta_days' and alias the result
        pl.col('delta_days').mean().alias('avg_delay_days')
    )
    .sort('avg_delay_days', descending=True) # Optional: Sort to see the highest delays first
)


Estos derivan en la tabla siguiente, donde se aprecia que Amapá, el Amazona y Roraima tienen los valores más altos:

In [69]:
delay_by_state.sort(
    # Column to sort by
    'avg_delay_days',
    # Parameter for descending order
    descending=True
)

state_name,geolocation_state,avg_delay_days
str,str,f64
"""Amapá""","""AP""",144.686782
"""Amazonas""","""AM""",40.685467
"""Roraima""","""RR""",37.089542
"""Acre""","""AC""",28.014091
"""Sergipe""","""SE""",19.219237
…,…,…
"""Santa Catarina""","""SC""",11.10427
"""Alagoas""","""AL""",11.093284
"""Distrito Federal""","""DF""",10.038074


Esta información se puede pasar a la función `.choropleth` de Plotly para construir un mapa. Cabe destaca que el archivo `geojson` es un archivo externo que contiene información de un sistema cartográfico que `Ploty` puede leer e interpretar para cronstruir la visualización:

In [71]:
# Crear figura con el mapa de Brasil y el choropleth
fig = px.choropleth(
    data_frame=delay_by_state,
    geojson=geojson,
    featureidkey='properties.UF',
    # featureidkey='properties.ESTADO',
    locations='geolocation_state',
    color='avg_delay_days',
    # https://plotly.com/python/builtin-colorscales/
    color_continuous_scale="bluyl",
    scope='south america',
    labels={'delta_days': 'Retraso (en días)'},
    width=800,
    height=400,
    title="Fig 8. Mapa del tiempo retraso promedio a nivel estatal"
)

# Actualizar diseño de la figura
fig.update_geos(
    showcountries=False,
    showcoastlines=True,
    showland=True,
    fitbounds='locations',
    visible=True
)

fig.update_layout(
    margin=dict(l=20, r=20, t=66, b=20),
    width=800,
    height=800,
)

# Mostrar figura
fig.show()


**Pregunta:**

* ¿Existe algun patrón en los estados donde se reportaron mayores retrazos?
* ¿La operación de Oilst debería tomar alguna medida a raíz de lo anterior en su opeación para alcanzar destinos en los estados más notorios del mapa?